|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Writing the kernel<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: coalesce the block-table gather<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
# find the repo root, wherever this notebook was opened from
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
import cudalib

This is PagedAttention's gather, with everything else taken away.

A table of rows in memory, an index saying which row each output needs, and a
sum over each gathered row. No softmax, no heads, no block size. Stage 07
builds the real one; this is the part of it that decides the speed.

In [2]:
### run this cell: the data, and the oracle

N_ROWS, ROW_LEN = 200_000, 128

table = torch.randn(N_ROWS, ROW_LEN, device='cuda')
index = torch.randperm(N_ROWS, device='cuda').to(torch.int32)  # a block table
out   = torch.empty(N_ROWS, device='cuda')

oracle = table[index.long()].sum(dim=1)
useful_bytes = N_ROWS * ROW_LEN * 4
print(f'{useful_bytes/1e6:.0f} MB of rows to gather')

102 MB of rows to gather


# Exercise 1: the obvious mapping, and what it costs

Give each output row one thread and let it walk its row. Write the kernel,
check it against the oracle, and measure what fraction of the card's
bandwidth you got.

In [3]:
NAIVE = r"""
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <torch/extension.h>

// One thread per row. It walks the whole row on its own.
__global__ void gather_naive(const float* __restrict__ table,
                             const int* __restrict__ index,
                             float* __restrict__ out,
                             const int n_rows, const int row_len) {
  const int r = blockIdx.x * blockDim.x + threadIdx.x;
  if (r >= n_rows) return;
  const float* src = table + (long)index[r] * row_len;
  float acc = 0.f;
  for (int d = 0; d < row_len; ++d) acc += src[d];
  out[r] = acc;
}

void gather(torch::Tensor table, torch::Tensor index, torch::Tensor out) {
  const int n_rows = index.numel(), row_len = table.size(1);
  const int threads = 256;
  gather_naive<<<(n_rows + threads - 1)/threads, threads, 0,
                 at::cuda::getCurrentCUDAStream()>>>(
      table.data_ptr<float>(), index.data_ptr<int>(), out.data_ptr<float>(),
      n_rows, row_len);
  C10_CUDA_KERNEL_LAUNCH_CHECK();
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("gather", &gather); }
"""

naive = cudalib.build_source('cc_gather_naive', NAIVE)

In [4]:
naive.gather(table, index, out)
print('correct:', torch.allclose(out, oracle, rtol=1e-3, atol=1e-2))

peak = cudalib.peak_bandwidth(fresh=True)
ms_naive = cudalib.bench_ms(lambda: naive.gather(table,index,out), best_of=3)
gb_naive = useful_bytes/(ms_naive*1e-3)/1e9

print(f'thread per row: {ms_naive:.3f} ms  {gb_naive:.0f} GB/s  {100*gb_naive/peak:.0f}% of peak')

correct: True


thread per row: 1.063 ms  96 GB/s  37% of peak


# Exercise 2: predict the fix before you write it

Do not measure first. Work out, from the sector arithmetic, how much faster a
coalesced version could be.

You will not get a single number, and that is the interesting part. Bound it
from both sides instead: the worst the naive mapping can do, and the best.
Where the measurement lands between them is a fact about the cache, and you
only get to learn it if you wrote the bounds down first.

In [5]:
floats_per_sector = 32 // 4

# At any one instruction the warp's 32 threads are ROW_LEN floats apart,
# so they touch 32 different sectors and use 1 float out of every 8.
worst_case = min(ROW_LEN, floats_per_sector)

# But the thread comes straight back for src[d+1], which is in the sector
# it just paid for. Walk the whole row and every fetched byte gets used.
# So the BYTES are not wasted; the REQUESTS are. 8 requests where 1 would
# do, and a cache that has to cover for it.
best_case = 1

print(f'stride between neighbouring threads: {ROW_LEN} floats')
print(f'speedup if nothing is cached:        {worst_case}x')
print(f'speedup if the cache catches it all: {best_case}x')
print(f'\nso: somewhere in 1x..{worst_case}x, and where it lands tells you\n'
      f'how much of the waste the cache absorbed.')

stride between neighbouring threads: 128 floats
speedup if nothing is cached:        8x
speedup if the cache catches it all: 1x

so: somewhere in 1x..8x, and where it lands tells you
how much of the waste the cache absorbed.


# Exercise 3: one warp per row

Same arithmetic, same answer. Change only which thread touches which byte:
the 32 lanes of a warp walk one row together, then combine their partial sums
with a shuffle.

In [6]:
COALESCED = r"""
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <torch/extension.h>

// One WARP per row. The 32 lanes walk the row together, so at every step they
// read 32 consecutive floats: one transaction instead of 32.
__global__ void gather_coalesced(const float* __restrict__ table,
                                 const int* __restrict__ index,
                                 float* __restrict__ out,
                                 const int n_rows, const int row_len) {
  const int warp = (blockIdx.x * blockDim.x + threadIdx.x) / 32;
  const int lane = threadIdx.x % 32;
  if (warp >= n_rows) return;

  const float* src = table + (long)index[warp] * row_len;

  // lane L takes elements L, L+32, L+64, ... so neighbouring lanes are
  // always on neighbouring addresses
  float acc = 0.f;
  for (int d = lane; d < row_len; d += 32) acc += src[d];

  // the row's total is now spread across 32 registers. Butterfly them
  // together; every lane in the mask must reach the shuffle.
  for (int o = 16; o > 0; o >>= 1) acc += __shfl_xor_sync(0xffffffff, acc, o);

  if (lane == 0) out[warp] = acc;
}

void gather(torch::Tensor table, torch::Tensor index, torch::Tensor out) {
  const int n_rows = index.numel(), row_len = table.size(1);
  const int threads = 256, warps_per_block = threads / 32;
  gather_coalesced<<<(n_rows + warps_per_block - 1)/warps_per_block, threads, 0,
                     at::cuda::getCurrentCUDAStream()>>>(
      table.data_ptr<float>(), index.data_ptr<int>(), out.data_ptr<float>(),
      n_rows, row_len);
  C10_CUDA_KERNEL_LAUNCH_CHECK();
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("gather", &gather); }
"""

fast = cudalib.build_source('cc_gather_fast', COALESCED)

In [7]:
fast.gather(table, index, out)
print('correct:', torch.allclose(out, oracle, rtol=1e-3, atol=1e-2))

ms_fast = cudalib.bench_ms(lambda: fast.gather(table,index,out), best_of=3)
gb_fast = useful_bytes/(ms_fast*1e-3)/1e9

print(f'thread per row: {ms_naive:.3f} ms  {gb_naive:6.0f} GB/s  {100*gb_naive/peak:3.0f}% of peak')
print(f'warp per row:   {ms_fast:.3f} ms  {gb_fast:6.0f} GB/s  {100*gb_fast/peak:3.0f}% of peak')
print(f'\nmeasured speedup:  {ms_naive/ms_fast:.2f}x')
print(f'predicted range:   {best_case}x .. {worst_case}x')

correct: True
thread per row: 1.063 ms      96 GB/s   37% of peak
warp per row:   0.326 ms     314 GB/s  120% of peak

measured speedup:  3.25x
predicted range:   1x .. 8x


### What you should be looking at

The measured speedup lands **inside** your bounds, not on either one. Around
3x, against a worst case of 8x and a best case of 1x.

That number is the answer to a question you could not have reasoned your way
to: the caches absorbed most of the wasted bytes, but not the wasted
requests. A thread that walks its row sequentially comes straight back for
the sector it just paid for, so almost nothing is thrown away. What it cannot
recover is having issued eight requests where one would do, and eight times
the latency to hide.

This is why the memory-transactions notebook got a clean 1.15 residual and
this did not. There, each thread read exactly one float and never came back,
so the sector model was the whole story. Here it is half of it.

The coalesced version also reports **more than 100% of peak**, which is not a
bug either: `useful_bytes` counts what the algorithm needed, not what the
hardware moved, and some gathered rows were still in L2 when a later warp
asked. A number over 100% is telling you the cache helped. Worth knowing, and
not worth hiding by quietly changing the denominator.

Now go and do it to the real kernel:

    ./vc guide 8b